# Homework A Coding Question

This notebook contains the source code for Question 4. It builds a visibility graph for a point robot among polygonal obstacles and solves the path with A*.

In [5]:
import heapq
import math
from pathlib import Path


EPS = 1e-9


def dist(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def orient(a, b, c):
    value = (b[0] - a[0]) * (c[1] - a[1]) - (b[1] - a[1]) * (c[0] - a[0])
    if abs(value) < EPS:
        return 0.0
    return value


def same_point(a, b):
    return dist(a, b) < 1e-8


def on_segment(a, b, p):
    if abs(orient(a, b, p)) > EPS:
        return False
    return (
        min(a[0], b[0]) - EPS <= p[0] <= max(a[0], b[0]) + EPS
        and min(a[1], b[1]) - EPS <= p[1] <= max(a[1], b[1]) + EPS
    )


def proper_segment_intersection(a, b, c, d):
    o1 = orient(a, b, c)
    o2 = orient(a, b, d)
    o3 = orient(c, d, a)
    o4 = orient(c, d, b)
    return o1 * o2 < -EPS and o3 * o4 < -EPS


def point_in_polygon_strict(p, polygon):
    inside = False
    n = len(polygon)
    for i in range(n):
        a = polygon[i]
        b = polygon[(i + 1) % n]
        if on_segment(a, b, p):
            return False
        crosses = (a[1] > p[1]) != (b[1] > p[1])
        if crosses:
            x_intersection = a[0] + (p[1] - a[1]) * (b[0] - a[0]) / (b[1] - a[1])
            if x_intersection > p[0] + EPS:
                inside = not inside
    return inside


def segment_visible(a, b, obstacles):
    midpoint = ((a[0] + b[0]) / 2.0, (a[1] + b[1]) / 2.0)
    for polygon in obstacles:
        if point_in_polygon_strict(midpoint, polygon):
            return False

        n = len(polygon)
        for i in range(n):
            c = polygon[i]
            d = polygon[(i + 1) % n]

            if proper_segment_intersection(a, b, c, d):
                return False

            for p in (c, d):
                if on_segment(a, b, p) and not same_point(p, a) and not same_point(p, b):
                    return False

    return True


def build_visibility_graph(start, goal, obstacles):
    nodes = [start, goal]
    for polygon in obstacles:
        nodes.extend(polygon)

    graph = {i: [] for i in range(len(nodes))}
    for i in range(len(nodes)):
        for j in range(i + 1, len(nodes)):
            if segment_visible(nodes[i], nodes[j], obstacles):
                weight = dist(nodes[i], nodes[j])
                graph[i].append((j, weight))
                graph[j].append((i, weight))
    return nodes, graph


def astar(nodes, graph, start_index, goal_index):
    open_heap = [(dist(nodes[start_index], nodes[goal_index]), 0.0, start_index)]
    parent = {start_index: None}
    g_cost = {start_index: 0.0}
    closed = set()

    while open_heap:
        _, cost_so_far, current = heapq.heappop(open_heap)
        if current in closed:
            continue
        closed.add(current)

        if current == goal_index:
            path = []
            while current is not None:
                path.append(current)
                current = parent[current]
            path.reverse()
            return path, cost_so_far

        for neighbor, edge_cost in graph[current]:
            new_cost = cost_so_far + edge_cost
            if neighbor not in g_cost or new_cost < g_cost[neighbor] - EPS:
                g_cost[neighbor] = new_cost
                parent[neighbor] = current
                priority = new_cost + dist(nodes[neighbor], nodes[goal_index])
                heapq.heappush(open_heap, (priority, new_cost, neighbor))

    raise RuntimeError("No path found")


def assert_path_is_collision_free(path_points, obstacles):
    for a, b in zip(path_points, path_points[1:]):
        if not segment_visible(a, b, obstacles):
            raise AssertionError(f"Path segment {a} -> {b} intersects an obstacle")


def plot_result(nodes, graph, path, obstacles, start, goal, filename):
    try:
        import matplotlib

        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        print("matplotlib is not installed; skipping plot.")
        return None

    fig, ax = plt.subplots(figsize=(7.5, 7.5))

    for polygon in obstacles:
        xs = [p[0] for p in polygon] + [polygon[0][0]]
        ys = [p[1] for p in polygon] + [polygon[0][1]]
        ax.fill(xs, ys, color="#d9d9d9", edgecolor="black", linewidth=1.5)

    for i, edges in graph.items():
        for j, _ in edges:
            if i < j:
                ax.plot(
                    [nodes[i][0], nodes[j][0]],
                    [nodes[i][1], nodes[j][1]],
                    color="#c7d7ee",
                    linewidth=0.7,
                    zorder=1,
                )

    path_points = [nodes[i] for i in path]
    ax.plot(
        [p[0] for p in path_points],
        [p[1] for p in path_points],
        color="#d62728",
        linewidth=3.0,
        marker="o",
        zorder=3,
    )
    ax.scatter([start[0]], [start[1]], color="#2ca02c", s=80, label="start", zorder=4)
    ax.scatter([goal[0]], [goal[1]], color="#9467bd", s=80, label="goal", zorder=4)
    ax.text(start[0] - 0.35, start[1] - 0.35, "start")
    ax.text(goal[0] + 0.1, goal[1] + 0.1, "goal")

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0, 9)
    ax.set_ylim(0, 8.5)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="lower right")
    ax.set_title("Visibility graph planner for a point robot")
    fig.tight_layout()
    fig.savefig(filename, dpi=200)
    plt.close(fig)
    return filename


def main():
    start = (2.6, 2.6)
    goal = (7.45, 7.30)

    obstacles = [
        [(0.45, 5.45), (1.05, 7.35), (3.45, 7.25), (4.00, 5.00), (2.05, 6.45)],
        [(5.25, 5.15), (6.05, 6.85), (7.30, 4.75)],
        [(2.05, 3.75), (4.05, 4.20), (3.25, 3.20), (3.75, 2.00)],
    ]

    nodes, graph = build_visibility_graph(start, goal, obstacles)
    path, length = astar(nodes, graph, 0, 1)
    path_points = [nodes[i] for i in path]
    assert_path_is_collision_free(path_points, obstacles)

    print("Obstacle vertices:")
    for obstacle_index, polygon in enumerate(obstacles, start=1):
        print(f"  obstacle {obstacle_index}: {polygon}")
    print()
    print(f"Number of roadmap nodes: {len(nodes)}")
    print(f"Number of roadmap edges: {sum(len(edges) for edges in graph.values()) // 2}")
    print("Shortest path:")
    for point in path_points:
        print(f"  ({point[0]:.2f}, {point[1]:.2f})")
    print(f"Total path length: {length:.6f}")

    script_dir = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
    output_file = script_dir / "homework4_q4_path.png"
    saved_file = plot_result(nodes, graph, path, obstacles, start, goal, output_file)
    if saved_file is not None:
        print(f"Saved plot to: {saved_file.resolve()}")


if __name__ == "__main__":
    main()


Obstacle vertices:
  obstacle 1: [(0.45, 5.45), (1.05, 7.35), (3.45, 7.25), (4.0, 5.0), (2.05, 6.45)]
  obstacle 2: [(5.25, 5.15), (6.05, 6.85), (7.3, 4.75)]
  obstacle 3: [(2.05, 3.75), (4.05, 4.2), (3.25, 3.2), (3.75, 2.0)]

Number of roadmap nodes: 14
Number of roadmap edges: 41
Shortest path:
  (2.60, 2.60)
  (2.05, 3.75)
  (4.00, 5.00)
  (6.05, 6.85)
  (7.45, 7.30)
Total path length: 7.822886
Saved plot to: A:\OneDrive - KU Leuven\Master\Robotics\Robotics_Enmin_Kaixi\Homework A\homework4_q4_path.png
